# 04 - Đánh Giá Mô Hình

Notebook này tính MAE, RMSE, MAPE và sMAPE cho các mô hình, sau đó lưu bảng kết quả và biểu đồ `y_true` vs `y_pred`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.evaluation import build_metrics_table, plot_predictions


## 1. Đọc kết quả dự báo

File `predictions.csv` được tạo bởi notebook 03. Mỗi cột mô hình phải được căn đúng với `y_true` trên tập test.

In [2]:
predictions_path = ROOT / 'results' / 'predictions.csv'
if not predictions_path.exists():
    raise FileNotFoundError('Hãy chạy notebooks/03_models.ipynb trước.')

predictions_df = pd.read_csv(predictions_path)
if 'datetime' in predictions_df.columns:
    predictions_df['datetime'] = pd.to_datetime(predictions_df['datetime'])

required = {'datetime', 'y_true'}
missing = required.difference(predictions_df.columns)
if missing:
    raise ValueError(f'predictions.csv đang thiếu cột: {sorted(missing)}')

model_columns = [
    column for column in predictions_df.columns
    if column not in {'datetime', 'row_index', 'y_true'}
]
print('Các mô hình được đánh giá:', model_columns)
display(predictions_df.head())


Các mô hình được đánh giá: ['Naive Forecast', 'Moving Average', 'Linear Regression', 'Random Forest', 'GRU']


,datetime,row_index,y_true,Naive Forecast,Moving Average,Linear Regression,Random Forest,GRU
0,2012-09-13 18:00:00,14629,782.22,782.22,312.810000,761.660833,737.777267,731.98150
1,2012-09-13 19:00:00,14630,674.00,782.22,312.810000,659.118242,600.572467,556.26074
2,2012-09-13 20:00:00,14631,463.00,674.00,314.435000,519.383005,455.403333,440.75390
3,2012-09-13 21:00:00,14632,317.00,463.00,314.143333,342.128309,350.343333,312.44230
4,2012-09-13 22:00:00,14633,251.00,317.00,314.601667,230.807725,225.516667,230.95169


## 2. Tính MAE, RMSE, MAPE và sMAPE

Vì `cnt` có thể bằng 0, MAPE bỏ qua các dòng có target bằng 0. sMAPE được dùng làm chỉ số phần trăm chính để tránh MAPE tăng bất thường.

In [3]:
predictions = {column: predictions_df[column] for column in model_columns}
metrics = build_metrics_table(predictions_df['y_true'], predictions)
metrics_path = ROOT / 'results' / 'metrics.csv'
metrics.to_csv(metrics_path, index=False)
print('Đã lưu:', metrics_path)
display(metrics.style.format({
    'mae': '{:.3f}',
    'rmse': '{:.3f}',
    'mape': '{:.2f}%',
    'smape': '{:.2f}%',
}))


Đã lưu: C:\Users\Dell\Desktop\time series\time-series-group-06\results\metrics.csv


,model,mae,rmse,mape,smape
0,GRU,28.393,43.979,27.69%,23.94%
1,Random Forest,34.330,57.025,23.81%,20.64%
2,Linear Regression,57.740,85.526,79.55%,43.23%
3,Naive Forecast,78.210,119.473,53.37%,45.41%
4,Moving Average,156.443,197.649,557.09%,78.45%


## 3. Vẽ `y_true` vs `y_pred`

Để biểu đồ dễ đọc, mặc định chỉ hiển thị 336 giờ đầu tiên của tập test, tương đương 14 ngày.

In [4]:
figure_path = ROOT / 'figures' / 'y_true_vs_y_pred.png'
plot_predictions(
    timestamps=predictions_df['datetime'],
    y_true=predictions_df['y_true'],
    predictions=predictions,
    output_path=figure_path,
    max_points=336,
)
print('Đã lưu:', figure_path)


Đã lưu: C:\Users\Dell\Desktop\time series\time-series-group-06\figures\y_true_vs_y_pred.png


## 4. Nội dung cần nhận xét trong báo cáo

Sau khi chạy xong, nhóm cần trả lời:

- Mô hình nào có MAE và RMSE thấp nhất?
- Mô hình nâng cao có tốt hơn Naive và Moving Average không?
- Mô hình dự báo tốt hay kém ở giờ cao điểm?
- Sai số có tăng vào ngày nghỉ, thời tiết xấu hoặc mùa có nhu cầu cao không?
- Kết quả có phù hợp với nhận xét từ EDA và ba bài báo không?